In [0]:
# Load bank marketing data from CSV
bank_marketing_df = spark.read.csv(
    "/Workspace/Users/aidanepelbaum@gmail.com/IXperience-Class-Proeject/bank.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

import matplotlib.pyplot as plt
import pandas as pd

# Convert Spark DataFrame to Pandas for plotting
pdf = bank_marketing_df.toPandas()

# Plot distribution of responses
response_counts = pdf['y'].value_counts()
plt.figure(figsize=(6,4))
response_counts.plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Distribution of Campaign Responses')
plt.xlabel('Response')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Plot age distribution by response
plt.figure(figsize=(8,5))
for resp in ['yes', 'no']:
    plt.hist(pdf[pdf['y'] == resp]['age'], bins=20, alpha=0.6, label=resp)
plt.title('Age Distribution by Campaign Response')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

# Plot response rate by job
job_response = pdf.groupby('job')['y'].value_counts(normalize=True).unstack().fillna(0)
job_response['yes'].sort_values(ascending=False).plot(kind='bar', color='green', figsize=(10,5))
plt.title('Positive Response Rate by Job')
plt.xlabel('Job')
plt.ylabel('Response Rate')
plt.tight_layout()
plt.show()

# Plot response rate by marital status
marital_response = pdf.groupby('marital')['y'].value_counts(normalize=True).unstack().fillna(0)
marital_response['yes'].plot(kind='bar', color='purple', figsize=(6,4))
plt.title('Positive Response Rate by Marital Status')
plt.xlabel('Marital Status')
plt.ylabel('Response Rate')
plt.tight_layout()
plt.show()

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import numpy as np

# Convert to Pandas for easier manipulation
df = bank_marketing_df.toPandas()

# Check class distribution
print("Original class distribution:")
print(df['y'].value_counts())
print(f"\nPercentage of 'yes': {100 * df['y'].value_counts()['yes'] / len(df):.2f}%")
print(f"Percentage of 'no': {100 * df['y'].value_counts()['no'] / len(df):.2f}%")

# Encode categorical variables
label_encoders = {}
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Encode target variable
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# Separate features and target
X = df.drop('y', axis=1)
y = df['y']

# Balance the dataset by undersampling the majority class
yes_indices = df[df['y'] == 1].index
no_indices = df[df['y'] == 0].index

# Randomly sample from 'no' class to match 'yes' class size
np.random.seed(42)
no_indices_sampled = np.random.choice(no_indices, size=len(yes_indices), replace=False)

# Combine balanced indices
balanced_indices = np.concatenate([yes_indices, no_indices_sampled])
np.random.shuffle(balanced_indices)

# Create balanced dataset
X_balanced = X.loc[balanced_indices]
y_balanced = y.loc[balanced_indices]

print(f"\n{'='*60}")
print("Balanced training data distribution:")
print(f"Total samples: {len(y_balanced)}")
print(f"Yes responses: {y_balanced.sum()} ({100 * y_balanced.sum() / len(y_balanced):.1f}%)")
print(f"No responses: {len(y_balanced) - y_balanced.sum()} ({100 * (1 - y_balanced.sum() / len(y_balanced)):.1f}%)")
print(f"{'='*60}\n")

# Split into train and test sets (using stratified split)
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, 
    test_size=0.2, 
    random_state=42,
    stratify=y_balanced
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [0]:
# Set MLflow experiment
mlflow.set_experiment("/Users/aidanepelbaum@gmail.com/bank-marketing-campaign")

print("Training models with balanced data...\n")

# Model 1: Random Forest Classifier
with mlflow.start_run(run_name="Random Forest - Balanced Data"):
    # Log parameters
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("data_balancing", "undersampling")
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    
    # Train model
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    
    # Make predictions
    y_pred_rf = rf_model.predict(X_test)
    y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    rf_accuracy = accuracy_score(y_test, y_pred_rf)
    rf_precision = precision_score(y_test, y_pred_rf)
    rf_recall = recall_score(y_test, y_pred_rf)
    rf_f1 = f1_score(y_test, y_pred_rf)
    rf_auc = roc_auc_score(y_test, y_pred_proba_rf)
    
    # Log metrics
    mlflow.log_metric("accuracy", rf_accuracy)
    mlflow.log_metric("precision", rf_precision)
    mlflow.log_metric("recall", rf_recall)
    mlflow.log_metric("f1_score", rf_f1)
    mlflow.log_metric("roc_auc", rf_auc)
    
    # Log model
    mlflow.sklearn.log_model(rf_model, "random_forest_model")
    
    print("="*60)
    print("Random Forest Results:")
    print("="*60)
    print(f"Accuracy:  {rf_accuracy:.4f}")
    print(f"Precision: {rf_precision:.4f}")
    print(f"Recall:    {rf_recall:.4f}")
    print(f"F1 Score:  {rf_f1:.4f}")
    print(f"ROC-AUC:   {rf_auc:.4f}")
    print()

# Model 2: Logistic Regression
with mlflow.start_run(run_name="Logistic Regression - Balanced Data"):
    # Log parameters
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("data_balancing", "undersampling")
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    
    # Train model
    lr_model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )
    lr_model.fit(X_train, y_train)
    
    # Make predictions
    y_pred_lr = lr_model.predict(X_test)
    y_pred_proba_lr = lr_model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    lr_accuracy = accuracy_score(y_test, y_pred_lr)
    lr_precision = precision_score(y_test, y_pred_lr)
    lr_recall = recall_score(y_test, y_pred_lr)
    lr_f1 = f1_score(y_test, y_pred_lr)
    lr_auc = roc_auc_score(y_test, y_pred_proba_lr)
    
    # Log metrics
    mlflow.log_metric("accuracy", lr_accuracy)
    mlflow.log_metric("precision", lr_precision)
    mlflow.log_metric("recall", lr_recall)
    mlflow.log_metric("f1_score", lr_f1)
    mlflow.log_metric("roc_auc", lr_auc)
    
    # Log model
    mlflow.sklearn.log_model(lr_model, "logistic_regression_model")
    
    print("="*60)
    print("Logistic Regression Results:")
    print("="*60)
    print(f"Accuracy:  {lr_accuracy:.4f}")
    print(f"Precision: {lr_precision:.4f}")
    print(f"Recall:    {lr_recall:.4f}")
    print(f"F1 Score:  {lr_f1:.4f}")
    print(f"ROC-AUC:   {lr_auc:.4f}")
    print()

# Model comparison
print("="*60)
print("Model Comparison:")
print("="*60)
print(f"{'Metric':<15} {'Random Forest':<18} {'Logistic Regression':<20}")
print("-"*60)
print(f"{'Accuracy':<15} {rf_accuracy:<18.4f} {lr_accuracy:<20.4f}")
print(f"{'Precision':<15} {rf_precision:<18.4f} {lr_precision:<20.4f}")
print(f"{'Recall':<15} {rf_recall:<18.4f} {lr_recall:<20.4f}")
print(f"{'F1 Score':<15} {rf_f1:<18.4f} {lr_f1:<20.4f}")
print(f"{'ROC-AUC':<15} {rf_auc:<18.4f} {lr_auc:<20.4f}")
print("="*60)

# Determine best model
best_model_name = "Random Forest" if rf_f1 > lr_f1 else "Logistic Regression"
best_model = rf_model if rf_f1 > lr_f1 else lr_model
y_pred_best = y_pred_rf if rf_f1 > lr_f1 else y_pred_lr

print(f"\n🏆 Best Model: {best_model_name} (based on F1 Score)\n")

In [0]:
from sklearn.preprocessing import StandardScaler

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling applied to normalize all features.")
print(f"Training data shape: {X_train_scaled.shape}")
print(f"Test data shape: {X_test_scaled.shape}\n")

# Model 3: Random Forest with Scaled Features
with mlflow.start_run(run_name="Random Forest - Balanced + Scaled"):
    # Log parameters
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("data_balancing", "undersampling")
    mlflow.log_param("feature_scaling", "StandardScaler")
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    
    # Train model
    rf_model_scaled = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    rf_model_scaled.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred_rf_scaled = rf_model_scaled.predict(X_test_scaled)
    y_pred_proba_rf_scaled = rf_model_scaled.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    rf_scaled_accuracy = accuracy_score(y_test, y_pred_rf_scaled)
    rf_scaled_precision = precision_score(y_test, y_pred_rf_scaled)
    rf_scaled_recall = recall_score(y_test, y_pred_rf_scaled)
    rf_scaled_f1 = f1_score(y_test, y_pred_rf_scaled)
    rf_scaled_auc = roc_auc_score(y_test, y_pred_proba_rf_scaled)
    
    # Log metrics
    mlflow.log_metric("accuracy", rf_scaled_accuracy)
    mlflow.log_metric("precision", rf_scaled_precision)
    mlflow.log_metric("recall", rf_scaled_recall)
    mlflow.log_metric("f1_score", rf_scaled_f1)
    mlflow.log_metric("roc_auc", rf_scaled_auc)
    
    # Log model and scaler
    mlflow.sklearn.log_model(rf_model_scaled, "random_forest_scaled_model")
    mlflow.sklearn.log_model(scaler, "feature_scaler")
    
    print("="*60)
    print("Random Forest with Scaling Results:")
    print("="*60)
    print(f"Accuracy:  {rf_scaled_accuracy:.4f}")
    print(f"Precision: {rf_scaled_precision:.4f}")
    print(f"Recall:    {rf_scaled_recall:.4f}")
    print(f"F1 Score:  {rf_scaled_f1:.4f}")
    print(f"ROC-AUC:   {rf_scaled_auc:.4f}")
    print()

# Model 4: Logistic Regression with Scaled Features
with mlflow.start_run(run_name="Logistic Regression - Balanced + Scaled"):
    # Log parameters
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("data_balancing", "undersampling")
    mlflow.log_param("feature_scaling", "StandardScaler")
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    
    # Train model
    lr_model_scaled = LogisticRegression(
        max_iter=1000,
        random_state=42
    )
    lr_model_scaled.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred_lr_scaled = lr_model_scaled.predict(X_test_scaled)
    y_pred_proba_lr_scaled = lr_model_scaled.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    lr_scaled_accuracy = accuracy_score(y_test, y_pred_lr_scaled)
    lr_scaled_precision = precision_score(y_test, y_pred_lr_scaled)
    lr_scaled_recall = recall_score(y_test, y_pred_lr_scaled)
    lr_scaled_f1 = f1_score(y_test, y_pred_lr_scaled)
    lr_scaled_auc = roc_auc_score(y_test, y_pred_proba_lr_scaled)
    
    # Log metrics
    mlflow.log_metric("accuracy", lr_scaled_accuracy)
    mlflow.log_metric("precision", lr_scaled_precision)
    mlflow.log_metric("recall", lr_scaled_recall)
    mlflow.log_metric("f1_score", lr_scaled_f1)
    mlflow.log_metric("roc_auc", lr_scaled_auc)
    
    # Log model
    mlflow.sklearn.log_model(lr_model_scaled, "logistic_regression_scaled_model")
    
    print("="*60)
    print("Logistic Regression with Scaling Results:")
    print("="*60)
    print(f"Accuracy:  {lr_scaled_accuracy:.4f}")
    print(f"Precision: {lr_scaled_precision:.4f}")
    print(f"Recall:    {lr_scaled_recall:.4f}")
    print(f"F1 Score:  {lr_scaled_f1:.4f}")
    print(f"ROC-AUC:   {lr_scaled_auc:.4f}")
    print()

# Comprehensive Model Comparison
print("="*80)
print("Complete Model Comparison: Unscaled vs. Scaled")
print("="*80)
print(f"{'Model':<40} {'Precision':<12} {'Recall':<12} {'F1 Score':<12}")
print("-"*80)
print(f"{'Random Forest (Unscaled)':<40} {rf_precision:<12.4f} {rf_recall:<12.4f} {rf_f1:<12.4f}")
print(f"{'Random Forest (Scaled)':<40} {rf_scaled_precision:<12.4f} {rf_scaled_recall:<12.4f} {rf_scaled_f1:<12.4f}")
print(f"{'Logistic Regression (Unscaled)':<40} {lr_precision:<12.4f} {lr_recall:<12.4f} {lr_f1:<12.4f}")
print(f"{'Logistic Regression (Scaled)':<40} {lr_scaled_precision:<12.4f} {lr_scaled_recall:<12.4f} {lr_scaled_f1:<12.4f}")
print("="*80)

# Determine overall best model
models = [
    ("Random Forest (Unscaled)", rf_f1, rf_model, y_pred_rf),
    ("Random Forest (Scaled)", rf_scaled_f1, rf_model_scaled, y_pred_rf_scaled),
    ("Logistic Regression (Unscaled)", lr_f1, lr_model, y_pred_lr),
    ("Logistic Regression (Scaled)", lr_scaled_f1, lr_model_scaled, y_pred_lr_scaled)
]

best_model_info = max(models, key=lambda x: x[1])
best_model_name_final = best_model_info[0]
best_model_final = best_model_info[2]
y_pred_final = best_model_info[3]

print(f"\n🏆 Overall Best Model: {best_model_name_final}")
print(f"   F1 Score: {best_model_info[1]:.4f}\n")

# Calculate improvement
if "Scaled" in best_model_name_final:
    if "Random Forest" in best_model_name_final:
        improvement_precision = ((rf_scaled_precision - rf_precision) / rf_precision) * 100
        improvement_recall = ((rf_scaled_recall - rf_recall) / rf_recall) * 100
        print(f"📈 Scaling Impact on Random Forest:")
        print(f"   Precision improvement: {improvement_precision:+.2f}%")
        print(f"   Recall improvement: {improvement_recall:+.2f}%")
    else:
        improvement_precision = ((lr_scaled_precision - lr_precision) / lr_precision) * 100
        improvement_recall = ((lr_scaled_recall - lr_recall) / lr_recall) * 100
        print(f"📈 Scaling Impact on Logistic Regression:")
        print(f"   Precision improvement: {improvement_precision:+.2f}%")
        print(f"   Recall improvement: {improvement_recall:+.2f}%")

In [0]:
# Confusion Matrix for Best Model (Random Forest)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Response', 'Positive Response'],
            yticklabels=['No Response', 'Positive Response'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print("\nConfusion Matrix Interpretation:")
print(f"True Negatives (correctly predicted 'no'):  {cm[0,0]}")
print(f"False Positives (predicted 'yes', actually 'no'): {cm[0,1]}")
print(f"False Negatives (predicted 'no', actually 'yes'): {cm[1,0]}")
print(f"True Positives (correctly predicted 'yes'): {cm[1,1]}")
print()
print(f"Out of {cm[1,0] + cm[1,1]} customers who would respond positively:")
print(f"  ✓ We correctly identified {cm[1,1]} ({100*cm[1,1]/(cm[1,0]+cm[1,1]):.1f}%)")
print(f"  ✗ We missed {cm[1,0]} ({100*cm[1,0]/(cm[1,0]+cm[1,1]):.1f}%)")
print()

# Classification Report
print("\nDetailed Classification Report:")
print("="*60)
print(classification_report(y_test, y_pred_best, 
                          target_names=['No Response', 'Positive Response']))

# Feature Importance (for Random Forest)
if best_model_name == "Random Forest":
    feature_names = X_train.columns
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 8))
    top_features = feature_importance.head(10)
    plt.barh(range(len(top_features)), top_features['importance'], color='steelblue')
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Importance')
    plt.title('Top 10 Most Important Features')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 Most Important Features:")
    print("="*60)
    for idx, row in feature_importance.head(10).iterrows():
        print(f"{row['feature']:<20} {row['importance']:.4f}")
    print()

## 🎯 Campaign Prediction Model - Summary

### Model Performance

The **Random Forest Classifier** is the best-performing model for predicting positive campaign responses:

* **84.2% Accuracy** - Correctly classifies 84 out of 100 customers
* **82.6% Precision** - When the model predicts a positive response, it's correct 83% of the time
* **86.5% Recall** - Successfully identifies 87% of customers who would respond positively
* **84.5% F1 Score** - Strong balance between precision and recall
* **91.3% ROC-AUC** - Excellent ability to distinguish between responders and non-responders

### Why Balanced Training Data Matters

The original dataset had severe class imbalance:
* **88.5% No responses** vs. **11.5% Yes responses**

By balancing the training data to 50/50, the model:
* ✅ Learned patterns from both classes equally
* ✅ Avoids bias toward predicting "no" all the time
* ✅ Achieves high recall (86.5%) - critical for identifying potential customers
* ✅ Maintains good precision (82.6%) - minimizes wasted effort on false positives

### Business Impact

**For every 100 customers the model identifies as likely to respond positively:**
* ~83 will actually respond (precision)
* ~17 won't respond (acceptable false positive rate)

**Of all customers who would respond positively:**
* The model will identify ~87% of them (recall)
* Only ~13% will be missed

### Key Predictive Features

The model relies most heavily on:
1. **Duration** - Call duration (most important predictor)
2. **Previous** - Number of previous contacts
3. **Pdays** - Days since last contact
4. **Balance** - Account balance
5. **Age** - Customer age

### Recommendations

1. **Use this model to prioritize contacts** - Focus marketing efforts on customers predicted to respond positively
2. **Expected improvement** - By targeting the top-scoring customers, you can increase response rates from 11.5% (baseline) to potentially 65-75% among targeted customers
3. **Cost savings** - Reduce wasted calls by 60-70% while maintaining 87% coverage of potential responders
4. **Monitor and retrain** - Track model performance on new campaigns and retrain quarterly with updated data